In [ ]:
# STEP 1: Ingest and inspect NSL-KDD
# Run this in Google Colab.

# Cell 1 — download directly (no manual upload needed, NSL-KDD is hosted publicly)
import urllib.request

train_url = "https://raw.githubusercontent.com/Mamcose/NSL-KDD-Network-Intrusion-Detection/master/NSL_KDD_Train.csv"
test_url = "https://raw.githubusercontent.com/Mamcose/NSL-KDD-Network-Intrusion-Detection/master/NSL_KDD_Test.csv"

urllib.request.urlretrieve(train_url, "nsl_kdd_train.csv")
urllib.request.urlretrieve(test_url, "nsl_kdd_test.csv")

# Cell 2 — load with proper column names (NSL-KDD ships without headers)
import pandas as pd

columns = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label"
]

# Try loading — if the source file already has headers, this block adapts
df = pd.read_csv("nsl_kdd_train.csv")
if df.columns[0] not in columns:
    # file had no header row, re-read with our column names
    df = pd.read_csv("nsl_kdd_train.csv", names=columns)

print("Shape:", df.shape)
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
print(df.head())

# Cell 3 — inspect the label column (will likely include attack subtypes, e.g. 'neptune', 'smurf', 'normal')
label_col = "label" if "label" in df.columns else df.columns[-1]
print(f"\nUsing label column: {label_col}")
print(df[label_col].value_counts())

# Cell 4 — map raw attack names into the 4 standard categories (DoS, Probe, R2L, U2R, normal)
# This mapping is the well-known NSL-KDD attack taxonomy
attack_map = {
    'normal': 'normal',
    'neptune': 'DoS', 'back': 'DoS', 'land': 'DoS', 'pod': 'DoS', 'smurf': 'DoS',
    'teardrop': 'DoS', 'mailbomb': 'DoS', 'apache2': 'DoS', 'processtable': 'DoS', 'udpstorm': 'DoS',
    'ipsweep': 'Probe', 'nmap': 'Probe', 'portsweep': 'Probe', 'satan': 'Probe', 'mscan': 'Probe', 'saint': 'Probe',
    'ftp_write': 'R2L', 'guess_passwd': 'R2L', 'imap': 'R2L', 'multihop': 'R2L', 'phf': 'R2L',
    'spy': 'R2L', 'warezclient': 'R2L', 'warezmaster': 'R2L', 'sendmail': 'R2L', 'named': 'R2L',
    'snmpgetattack': 'R2L', 'snmpguess': 'R2L', 'xlock': 'R2L', 'xsnoop': 'R2L', 'worm': 'R2L',
    'buffer_overflow': 'U2R', 'loadmodule': 'U2R', 'perl': 'U2R', 'rootkit': 'U2R',
    'httptunnel': 'U2R', 'ps': 'U2R', 'sqlattack': 'U2R', 'xterm': 'U2R',
}

df["label_clean"] = df[label_col].astype(str).str.strip().str.replace(".", "", regex=False)
df["attack_category"] = df["label_clean"].map(attack_map).fillna("unknown")

print("\nAttack category distribution:")
print(df["attack_category"].value_counts())
print("\nAny unmapped labels? (should be empty or near-empty):")
print(df[df["attack_category"] == "unknown"]["label_clean"].value_counts())

Shape: (125973, 42)

Columns: ['duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes', 'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in', 'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations', 'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login', 'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate', 'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate', 'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count', 'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate', 'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate', 'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label']

First 5 rows:
   duration protocol_type   service flag  src_bytes  dst_bytes  land  \
0         0           tcp  ftp_data   SF        491          0     0   
1         0           udp     other   SF        146          0     0   
2

In [ ]:
# STEP 2: Normalization
# Simulates ingestion from 3 "fragmented" sources by splitting NSL-KDD's
# 41 features into logical groups, then mapping each group into one
# common normalized schema -- the way Cosmos would normalize firewall,
# IDS, and endpoint telemetry into a single correlatable record.
#
# Run this in the same Colab session, after step 1.

import pandas as pd

# --- "Source A": connection metadata (sounds like firewall/network log) ---
source_a_fields = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land"
]

# --- "Source B": content/behavioral features (sounds like host/endpoint log) ---
source_b_fields = [
    "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell",
    "su_attempted", "num_root", "num_file_creations", "num_shells",
    "num_access_files", "is_guest_login"
]

# --- "Source C": traffic statistics (sounds like IDS/flow-analysis log) ---
source_c_fields = [
    "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate",
    "same_srv_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_serror_rate"
]

def normalize_record(row, original_index):
    """
    Maps a single raw NSL-KDD row into one common normalized schema,
    as if it had been ingested from three separate sources and joined.

    original_index is carried through so any normalized record can be
    traced back to its exact source row -- without this, there's no way
    to verify a transformed record against its raw input, which is a real
    data-lineage problem any production pipeline needs to avoid.
    """
    normalized = {
        # common schema fields every source maps into
        "conn_protocol": row["protocol_type"],
        "conn_service": row["service"],
        "conn_flag_status": row["flag"],
        "bytes_sent": row["src_bytes"],
        "bytes_received": row["dst_bytes"],

        "auth_failed_logins": row["num_failed_logins"],
        "auth_logged_in": bool(row["logged_in"]),
        "host_compromise_indicators": row["num_compromised"] + row["root_shell"] + row["su_attempted"],
        "guest_login_flag": bool(row["is_guest_login"]),

        "conn_count_same_host": row["count"],
        "conn_error_rate": row["serror_rate"],
        "dst_host_conn_count": row["dst_host_count"],
        "dst_host_error_rate": row["dst_host_serror_rate"],

        # ground truth, carried through for validation later -- NOT shown to the LLM
        "_ground_truth_category": row["attack_category"],
        # traceability back to the exact source row -- NOT shown to the LLM
        "_source_row_index": original_index,
    }
    return normalized

# Apply normalization to a manageable sample for the demo (3 per category,
# so ~15 records total -- keeps total Gemini API calls low given free-tier
# rate limits, while still covering every attack category)
sample = (
    df.groupby("attack_category", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), 3), random_state=42))
    .reset_index(drop=True)
)

normalized_records = [normalize_record(row, original_index=idx) for idx, row in sample.iterrows()]
normalized_df = pd.DataFrame(normalized_records)
print(normalized_df.head())

print("Normalized sample shape:", normalized_df.shape)
print("\nGround truth distribution in this sample:")
print(normalized_df["_ground_truth_category"].value_counts())
print("\nFirst 5 normalized records (with traceable source row index):")
print(normalized_df[["_source_row_index", "_ground_truth_category"]].head())

# Sanity check: did normalization lose or corrupt anything?
print("\nNulls introduced during normalization:")
print(normalized_df.isnull().sum())

# --- Lookup helper: trace any normalized record back to its exact raw row ---
def show_raw_row_for(record_index):
    """Given an index into normalized_records, print the exact original
    row from df that it came from -- so you can manually verify a
    narrative's numbers against the real source row, not a guess."""
    src_idx = normalized_records[record_index]["_source_row_index"]
    print(f"normalized_records[{record_index}] came from df row index {src_idx}:\n")
    print(df.loc[src_idx])

# Example: trace record 0 back to its real raw row
show_raw_row_for(0)

  conn_protocol conn_service conn_flag_status  bytes_sent  bytes_received  \
0           tcp      private               S0           0               0   
1           tcp      private               S0           0               0   
2           tcp       finger               S0           0               0   
3          icmp        eco_i               SF           8               0   
4          icmp        eco_i               SF           8               0   

   auth_failed_logins  auth_logged_in  host_compromise_indicators  \
0                   0           False                           0   
1                   0           False                           0   
2                   0           False                           0   
3                   0           False                           0   
4                   0           False                           0   

   guest_login_flag  conn_count_same_host  conn_error_rate  \
0             False                   229              1.0  

/tmp/ipykernel_6562/4225275674.py:70: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), 3), random_state=42))


In [ ]:
# STEP 3: PromptOps - generate human-readable threat narratives via LLM
# STEP 4: Output validation - check narratives for hallucination / faithfulness
#
# Run this in the same Colab session, after step 1 and step 2.
#
# You'll need an API key. In Colab: click the key icon in the left sidebar
# (Secrets), add a secret named GEMINI_API_KEY, then run this.
# Get a free key at https://aistudio.google.com/apikey

# Cell 1 — install + auth
!pip install -q -U google-genai

from google.colab import userdata
from google import genai

client = genai.Client(api_key=userdata.get("GEMINI_API_KEY"))

# Cell 2 — build the prompt (deliberately do NOT include ground truth)
def build_prompt(record):
    # strip ground truth before sending to the LLM -- it should never see the answer
    visible_fields = {k: v for k, v in record.items() if not k.startswith("_")}
    return f"""You are a security analyst assistant. Given the following normalized
network connection record, write a 2-3 sentence plain-English summary describing
whether this looks suspicious, and why. Base your answer ONLY on the fields provided.
Do not invent details that aren't in the data.

Record:
{visible_fields}

Summary:"""

def get_narrative(record, max_retries=5):
    import time
    prompt = build_prompt(record)
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
            )
            return response.text.strip()
        except Exception as e:
            if "RESOURCE_EXHAUSTED" in str(e) or "429" in str(e):
                wait = 15  # free tier is 5 requests/minute, so wait it out
                print(f"  Rate limited, waiting {wait}s before retry ({attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError(f"Failed after {max_retries} retries due to rate limiting.")

# Cell 3 — run PromptOps over the normalized sample
# Free tier = 5 requests/minute, hard limit, regardless of sample size.
# This paces strictly to 1 request per 13s by tracking actual elapsed time,
# so it self-corrects even if a call itself takes a few seconds.
import time

results = []
last_call_time = None

for i, record in enumerate(normalized_records):
    if last_call_time is not None:
        elapsed = time.time() - last_call_time
        wait_needed = 13 - elapsed
        if wait_needed > 0:
            time.sleep(wait_needed)

    last_call_time = time.time()
    narrative = get_narrative(record)

    results.append({
        "record_index": i,
        "ground_truth": record["_ground_truth_category"],
        "source_row_index": record["_source_row_index"],
        "narrative": narrative,
        "raw_record": {k: v for k, v in record.items() if not k.startswith("_")},
    })
    print(f"[{i}] truth={record['_ground_truth_category']:10s} -> {narrative}\n")

# Cell 4 — OUTPUT VALIDATION: this is your QA layer
#
# Two checks:
#  (a) Directional accuracy: does the narrative's verdict (suspicious vs not)
#      roughly agree with ground truth? -- catches false positives/negatives
#  (b) Faithfulness: does the narrative reference values that are NOT actually
#      in the record? -- catches hallucination

def check_directional_accuracy(narrative, ground_truth):
    """
    Looks for an explicit verdict phrase rather than a bare keyword match,
    so negated statements like "does not appear suspicious" are read correctly.
    """
    text = narrative.lower()

    negative_verdict_phrases = [
        "does not appear suspicious", "not appear suspicious",
        "does not appear malicious", "no indicators of", "does not appear immediately suspicious",
        "not suspicious", "appears legitimate", "consistent with legitimate",
        "suggesting legitimate activity", "suggest legitimate activity",
    ]
    positive_verdict_phrases = [
        "highly suspicious", "appears suspicious", "is suspicious",
        "looks suspicious", "moderately suspicious", "looks highly suspicious",
        "malicious", "denial-of-service", "port scan", "brute-force",
        "reconnaissance", "scanning activity",
    ]

    has_negative = any(p in text for p in negative_verdict_phrases)
    has_positive = any(p in text for p in positive_verdict_phrases)

    # If both fire (e.g. "this is not malicious, but X is suspicious"), trust
    # whichever phrase appears first as the overall verdict.
    if has_negative and has_positive:
        first_neg = min(text.find(p) for p in negative_verdict_phrases if p in text)
        first_pos = min(text.find(p) for p in positive_verdict_phrases if p in text)
        flagged_suspicious = first_pos < first_neg
    else:
        flagged_suspicious = has_positive and not has_negative

    is_actually_attack = ground_truth != "normal"
    if flagged_suspicious == is_actually_attack:
        return "MATCH"
    elif flagged_suspicious and not is_actually_attack:
        return "FALSE_POSITIVE"
    else:
        return "FALSE_NEGATIVE"

def check_faithfulness(narrative, raw_record):
    """
    Checks whether numbers mentioned in the narrative trace back to the raw
    record -- now accounts for rate fields stored as decimals (e.g. 1.0)
    but described in the narrative as percentages (e.g. "100%").
    """
    import re
    numbers_in_narrative = set(re.findall(r"\b\d+\.?\d*\b", narrative))

    numbers_in_record = set()
    for v in raw_record.values():
        numbers_in_record.add(str(v))
        # if it's a rate between 0 and 1, also add its percentage form
        try:
            fv = float(v)
            if 0 <= fv <= 1:
                pct = fv * 100
                numbers_in_record.add(str(int(pct)) if pct == int(pct) else str(pct))
        except (ValueError, TypeError):
            pass

    unsupported_numbers = numbers_in_narrative - numbers_in_record
    return list(unsupported_numbers)

validation_rows = []
for r in results:
    accuracy_flag = check_directional_accuracy(r["narrative"], r["ground_truth"])
    unsupported = check_faithfulness(r["narrative"], r["raw_record"])
    validation_rows.append({
        "record_index": r["record_index"],
        "ground_truth": r["ground_truth"],
        "accuracy_flag": accuracy_flag,
        "unsupported_numbers_in_narrative": unsupported,
        "narrative": r["narrative"],
    })

validation_df = pd.DataFrame(validation_rows)

print("\n" + "=" * 60)
print("VALIDATION SUMMARY")
print("=" * 60)
print("\nAccuracy flag distribution:")
print(validation_df["accuracy_flag"].value_counts())

print("\nFalse positives (flagged suspicious but ground truth = normal):")
print(validation_df[validation_df["accuracy_flag"] == "FALSE_POSITIVE"][["record_index", "narrative"]])

print("\nFalse negatives (missed an actual attack):")
print(validation_df[validation_df["accuracy_flag"] == "FALSE_NEGATIVE"][["record_index", "ground_truth", "narrative"]])

print("\nRecords with unsupported numbers in narrative (possible hallucination):")
print(validation_df[validation_df["unsupported_numbers_in_narrative"].apply(len) > 0])

[0] truth=DoS        -> This record looks highly suspicious. It details a high volume of connection attempts (229 from the same host, 255 to the destination) all resulting in an S0 flag, indicating failed connections with no data exchanged. The 100% error rate for these connections strongly suggests activities like port scanning or a denial-of-service attempt targeting the 'private' service.

[1] truth=DoS        -> This record is suspicious. It shows 280 failed TCP connection attempts (S0 flag, 100% error rate) from the same source to a single destination, which also experiences a high overall connection error rate (99%). This pattern of repeated, unsuccessful connections could indicate a port scan or a targeted denial-of-service attempt.

[2] truth=DoS        -> This network connection record looks suspicious. There were numerous failed attempts to connect to the 'finger' service, indicated by a 100% error rate across 68 connections to the same host and 255 connections to the destina

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# STEP 6: Evaluate LLM outputs using DeepEval
#
# Run this in the same Colab session, after step 3 has finished and
# `results` is in memory. This replaces our hand-rolled faithfulness
# checker with DeepEval -- a production-grade LLM evaluation framework.
#
# DeepEval uses an LLM-as-judge pattern: it sends the original input,
# the LLM's output, and the expected answer to a judge model (Gemini/GPT)
# which scores quality on each metric from 0 to 1.
#
# WHY THIS MATTERS FOR THE INTERVIEW:
# We built our own faithfulness checker from scratch and found two bugs:
# (1) negation handling in keyword matching
# (2) rate-vs-percentage mismatch in numeric comparison
# DeepEval handles both of these correctly because it uses semantic
# understanding rather than string matching -- this is why production
# systems use frameworks rather than hand-rolled heuristics.

# Cell 1 — install
!pip install deepeval -q

# Cell 2 — configure DeepEval to use Gemini
# DeepEval normally defaults to OpenAI, but supports custom LLM judges.
# We'll use our existing Gemini client as the judge model.
import os
from google.colab import userdata

# DeepEval needs OPENAI_API_KEY by default -- we override with a custom model below
# so this won't be used, but the library checks for it at import time
os.environ["OPENAI_API_KEY"] = "not-used-we-use-custom-model"

from deepeval import evaluate
from deepeval.metrics import (
    FaithfulnessMetric,
    AnswerRelevancyMetric,
    GEval,
)
from deepeval.models.base_model import DeepEvalBaseLLM
from deepeval.test_case import LLMTestCase
try:
    from deepeval.test_case import SingleTurnParams as EvalParams
except ImportError:
    from deepeval.test_case import LLMTestCaseParams as EvalParams
from google import genai as google_genai

# Cell 3 — wrap Gemini as a DeepEval-compatible judge model
class GeminiJudge(DeepEvalBaseLLM):
    """Wraps our existing Gemini client so DeepEval can use it as a
    judge model for scoring LLM output quality."""

    def __init__(self):
        self.client = google_genai.Client(
            api_key=userdata.get("GEMINI_API_KEY")
        )

    def load_model(self):
        return self.client

    def generate(self, prompt: str) -> str:
        response = self.client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt,
        )
        return response.text.strip()

    async def a_generate(self, prompt: str) -> str:
        return self.generate(prompt)

    def get_model_name(self) -> str:
        return "gemini-2.5-flash"

judge = GeminiJudge()

# Cell 4 — build DeepEval test cases from our existing results
# Each test case needs:
#   input     = what we sent to the LLM (the normalized record as a string)
#   actual_output = what the LLM said (the narrative)
#   retrieval_context = the source data the LLM was supposed to reason from
#   expected_output   = what we know the correct answer is (ground truth category)

def record_to_string(raw_record):
    """Converts a normalized record dict into a readable string for DeepEval."""
    return "\n".join(f"{k}: {v}" for k, v in raw_record.items())

test_cases = []
for r in results:
    tc = LLMTestCase(
        input=record_to_string(r["raw_record"]),
        actual_output=r["narrative"],
        retrieval_context=[record_to_string(r["raw_record"])],
        expected_output=f"This connection is classified as: {r['ground_truth']}",
    )
    test_cases.append(tc)

print(f"Built {len(test_cases)} DeepEval test cases")
print(f"\nExample test case input:\n{test_cases[0].input}")
print(f"\nExpected output: {test_cases[0].expected_output}")
print(f"\nActual output: {test_cases[0].actual_output}")

# Cell 5 — DeepEval metrics, one record, one metric at a time
import time

# Evaluate ONE record only, ONE metric at a time, with 65s between each.
# DeepEval makes 2-3 internal API calls per metric (claim extraction +
# per-claim verification), so the per-minute quota exhausts much faster
# than a single Gemini call would. 65s fully resets the window.

faithfulness_metric = FaithfulnessMetric(
    threshold=0.7, model=judge, include_reason=True,
)
relevancy_metric = AnswerRelevancyMetric(
    threshold=0.7, model=judge, include_reason=True,
)
correctness_metric = GEval(
    name="Threat Classification Correctness",
    criteria=(
        "Evaluate whether the narrative correctly identifies the threat category. "
        "A 'normal' record should not be described as suspicious. "
        "A DoS/Probe/R2L/U2R record should be flagged as suspicious or concerning."
    ),
    evaluation_params=[
        EvalParams.INPUT,
        EvalParams.ACTUAL_OUTPUT,
        EvalParams.EXPECTED_OUTPUT,
    ],
    model=judge,
    threshold=0.7,
)

tc = test_cases[0]
truth = results[0]["ground_truth"]

print(f"Evaluating single record: ground_truth={truth}")
print(f"\nInput:\n{tc.input}")
print(f"\nActual output:\n{tc.actual_output}")

metrics_to_run = [
    ("FaithfulnessMetric", faithfulness_metric),
    ("AnswerRelevancyMetric", relevancy_metric),
    ("Threat Classification Correctness", correctness_metric),
]

eval_results = {}

for name, metric in metrics_to_run:
    print(f"\n{'='*60}")
    print(f"Running: {name}")
    print("Waiting 65s for rate limit to reset...")
    time.sleep(65)
    try:
        metric.measure(tc)
        eval_results[name] = {
            "score": round(metric.score, 2),
            "passed": metric.score >= metric.threshold,
            "reason": metric.reason,
        }
        status = "PASS" if metric.score >= metric.threshold else "FAIL"
        print(f"Score: {metric.score:.2f} [{status}]")
        print(f"Reason: {metric.reason}")
    except Exception as e:
        print(f"Error: {e}")
        eval_results[name] = {"score": None, "passed": None, "reason": str(e)}

print(f"\n{'='*60}")
print("DEEPEVAL SUMMARY")
print(f"{'='*60}")
print(f"Ground truth: {truth}")
print(f"Narrative: {tc.actual_output[:120]}...\n")
for name, r in eval_results.items():
    if r["score"] is not None:
        status = "PASS" if r["passed"] else "FAIL"
        print(f"[{status}]  {name}: {r['score']}")
        print(f"        {r['reason']}\n")

print("\nKEY INSIGHT:")
print("Each DeepEval metric makes 2-3 internal API calls -- claim extraction")
print("then per-claim verification. Production use needs a paid API tier or")
print("a locally-hosted judge model to avoid free-tier quota exhaustion.")

Built 5 DeepEval test cases

Example test case input:
conn_protocol: tcp
conn_service: private
conn_flag_status: S0
bytes_sent: 0
bytes_received: 0
auth_failed_logins: 0
auth_logged_in: False
host_compromise_indicators: 0
guest_login_flag: False
conn_count_same_host: 229
conn_error_rate: 1.0
dst_host_conn_count: 255
dst_host_error_rate: 1.0

Expected output: This connection is classified as: DoS

Actual output: This record looks highly suspicious. It details a high volume of connection attempts (229 from the same host, 255 to the destination) all resulting in an S0 flag, indicating failed connections with no data exchanged. The 100% error rate for these connections strongly suggests activities like port scanning or a denial-of-service attempt targeting the 'private' service.
Evaluating single record: ground_truth=DoS

Input:
conn_protocol: tcp
conn_service: private
conn_flag_status: S0
bytes_sent: 0
bytes_received: 0
auth_failed_logins: 0
auth_logged_in: False
host_compromise_indicator

Output()

Score: 1.00 [PASS]
Reason: The score is 1.00 because the actual output is perfectly faithful to the retrieval context. Great job!

Running: AnswerRelevancyMetric
Waiting 65s for rate limit to reset...


Output()

Score: 1.00 [PASS]
Reason: The score is 1.00 because the output is perfectly relevant and directly addresses the input without any extraneous information. Excellent job!

Running: Threat Classification Correctness
Waiting 65s for rate limit to reset...


Output()

Score: 1.00 [PASS]
Reason: The input data, characterized by high connection counts (229 from same host, 255 to destination), S0 flag status, and 100% error rates, indicates a DoS/Probe threat category. The actual output accurately reflects this by stating the record is 'highly suspicious' and strongly suggesting 'port scanning or a denial-of-service attempt,' which aligns perfectly with the expected 'DoS' classification.

DEEPEVAL SUMMARY
Ground truth: DoS
Narrative: This record looks highly suspicious. It details a high volume of connection attempts (229 from the same host, 255 to the...

[PASS]  FaithfulnessMetric: 1.0
        The score is 1.00 because the actual output is perfectly faithful to the retrieval context. Great job!

[PASS]  AnswerRelevancyMetric: 1.0
        The score is 1.00 because the output is perfectly relevant and directly addresses the input without any extraneous information. Excellent job!

[PASS]  Threat Classification Correctness: 1.0
        The input data, c

In [ ]:
# STEP 5: Manual faithfulness review
#
# Run this in the same Colab session, AFTER step3_4 has finished and
# `results` exists in memory.
#
# Purpose: print each record's raw normalized fields directly next to the
# LLM's narrative, with every number in the narrative highlighted, so you
# can eyeball-verify whether each numeric claim actually traces back to a
# real field value -- this is "checking the checker": it gives you a
# ground truth for whether the automated faithfulness function itself can
# be trusted, the same way you'd manually verify a few rows before trusting
# an automated test suite's output.

import re

def highlight_numbers(text):
    """Wraps every number in the narrative with >>> <<< so it's easy to
    spot-check against the raw record fields below it."""
    return re.sub(r"\b\d+\.?\d*\b", lambda m: f">>>{m.group()}<<<", text)

def percent_form(value):
    """If value looks like a 0-1 rate, also show its percentage form,
    since narratives often describe rates as percentages."""
    try:
        fv = float(value)
        if 0 <= fv <= 1:
            pct = fv * 100
            return f"{value}  (as %: {int(pct) if pct == int(pct) else pct})"
    except (ValueError, TypeError):
        pass
    return str(value)

print("=" * 90)
print("MANUAL FAITHFULNESS REVIEW")
print("Every number in the narrative is wrapped in >>> <<<")
print("Check each one against the raw record fields printed below it.")
print("=" * 90)

for r in results:
    print(f"\n{'-' * 90}")
    print(f"Record [{r['record_index']}]   ground truth = {r['ground_truth']}   "
          f"(source = df.loc[{r['source_row_index']}] -- look this up directly to double check)")
    print(f"{'-' * 90}")

    print("\nRAW NORMALIZED RECORD:")
    for k, v in r["raw_record"].items():
        print(f"   {k:32s} = {percent_form(v)}")

    print("\nNARRATIVE (numbers flagged for checking):")
    print("   " + highlight_numbers(r["narrative"]))

    print("\n   >> Manually confirm: does every >>>number<<< above appear")
    print("      (or have an obvious percentage-equivalent) in the raw")
    print("      record fields? If not, that's a real hallucination.")

print("\n" + "=" * 90)
print("Done. Scroll up and check each record's flagged numbers by eye.")
print("=" * 90)

MANUAL FAITHFULNESS REVIEW
Every number in the narrative is wrapped in >>> <<<
Check each one against the raw record fields printed below it.

------------------------------------------------------------------------------------------
Record [0]   ground truth = DoS   (source = df.loc[0] -- look this up directly to double check)
------------------------------------------------------------------------------------------

RAW NORMALIZED RECORD:
   conn_protocol                    = tcp
   conn_service                     = private
   conn_flag_status                 = S0
   bytes_sent                       = 0  (as %: 0)
   bytes_received                   = 0  (as %: 0)
   auth_failed_logins               = 0  (as %: 0)
   auth_logged_in                   = False  (as %: 0)
   host_compromise_indicators       = 0  (as %: 0)
   guest_login_flag                 = False  (as %: 0)
   conn_count_same_host             = 229
   conn_error_rate                  = 1.0  (as %: 100)
   dst_host_con